# Romansh retrieval playground
Run the cells in order in JupyterLab from a clone of this repository. Install dependencies in the active environment with `python -m pip install -r requirements.txt jupyterlab ipywidgets matplotlib` from the repository root.

Start by exploring the **La Quotidiana** cleaning audit, then try retrieval on aligned **Mediomatix** schoolbook segments, and finally compare the team's recorded held-out results. The newspaper corpus is a separate data-quality study. Its articles were not used for retrieval training or evaluation.

The headings and instructions render on GitHub. Open in JupyterLab and run all cells to use the controls. The audit and results use committed files. Live retrieval needs the Mediomatix validation split. TF-IDF runs on CPU; dense models are optional.

In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src" / "romansh_retrieval").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from a clone of the project repository.")
sys.path.insert(0, str(ROOT / "src"))

from functools import lru_cache
import ipywidgets as widgets
import numpy as np
import pandas as pd
from IPython.display import display
from romansh_retrieval.config import IDIOMS, DEFAULT_MODEL, XLM_R_MODEL
from romansh_retrieval.data import load_split, make_pairs
from romansh_retrieval.methods import tfidf_vectors, load_embedding_model, embedding_vectors

## 1. Follow one query through the pipeline

Given a passage in one idiom, can we find its aligned counterpart in another? The pipeline keeps three questions separate: What noise does the newspaper audit reveal? How well does retrieval rank aligned schoolbook text? How does fine-tuning change the recorded test results?

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Markdown
from romansh_retrieval.overlap import add_overlap_buckets
from romansh_retrieval.eval import ranks_from_vectors

fig, ax = plt.subplots(figsize=(11, 2.7))
ax.set(xlim=(0, 1), ylim=(0, 1))
ax.axis("off")
steps = ["Aligned\nMediomatix rows", "Select direction\nand sample candidates",
         "Encode query\nand candidates", "Rank by cosine\nsimilarity", "Score the aligned\nmatch"]
for i, label in enumerate(steps):
    x = 0.10 + 0.20 * i
    ax.text(x, 0.65, label, ha="center", va="center", fontsize=10,
            bbox=dict(boxstyle="round,pad=0.55", fc="#e0f2fe", ec="#0369a1"))
    if i:
        ax.annotate("", (x - 0.087, 0.65), (x - 0.112, 0.65),
                    arrowprops=dict(arrowstyle="->", color="#0369a1"))
ax.text(0.5, 0.21, "Separate analyses: La Quotidiana cleaning audit • Held-out model comparison",
        ha="center", color="#334155", fontsize=10)
plt.show()

## 2. Explore the newspaper audit
Filter the manually reviewed sample by cleaning flag or variety. The counts describe the audit sample, which was stratified by flag.

In [ ]:
audit = pd.read_csv(ROOT / "data_quality" / "audit_sample_labeled.csv")

def audit_view(flag, variety, verdict):
    selected = audit
    if flag != "All":
        selected = selected[selected["flag"].eq(flag)]
    if variety != "All":
        selected = selected[selected["variety"].eq(variety)]
    if verdict != "All":
        selected = selected[selected["verdict"].eq(verdict)]
    print(f"{len(selected)} audited articles match")
    display(selected["verdict"].value_counts().rename("articles").to_frame())
    with pd.option_context("display.max_colwidth", 240):
        display(selected[["post_id", "variety", "flag", "verdict", "text"]].head(12))

display(widgets.interactive(
    audit_view,
    flag=widgets.Dropdown(options=["All", *sorted(audit["flag"].dropna().unique())]),
    variety=widgets.Dropdown(options=["All", *sorted(audit["variety"].dropna().unique())]),
    verdict=widgets.Dropdown(options=["All", *sorted(audit["verdict"].dropna().unique())]),
))

In [ ]:
audit_counts = pd.crosstab(audit["flag"], audit["verdict"])
ax = audit_counts.plot.barh(stacked=True, figsize=(8, 5), colormap="Set2")
ax.set(title="Manual audit sample by cleaning flag", xlabel="Audited entries", ylabel="")
ax.legend(title="Reviewer verdict")
plt.tight_layout()
plt.show()

## 3. Inspect aligned pairs and lexical overlap

The gold match is the target from the same aligned row. Lexical overlap is the Jaccard similarity of the source and target character n-gram sets. It measures shared spelling. Cosine similarity is the model score used to rank candidates.

The cells use the real validation split if it is present. Otherwise they use a small synthetic sample for an offline demonstration. In the synthetic sample, each column is a deliberately altered version of the same example text. It is not annotated Romansh and cannot support model comparisons.

In [ ]:
validation_path = ROOT / "data" / "raw" / "validation.parquet"
topics = ["mountain trail", "fresh bread", "school books", "quiet river",
          "winter snow", "train station", "village market", "morning coffee"]
variants = [("", ""), ("la ", ""), ("", " nova"), ("in ", ""), ("", " buna")]
demo_rows = {
    idiom: [prefix + topic + suffix for topic in topics]
    for idiom, (prefix, suffix) in zip(IDIOMS, variants)
}
data_label = "Mediomatix validation" if validation_path.is_file() else "Synthetic offline demo"
print(data_label)

@lru_cache(maxsize=1)
def validation_data():
    if validation_path.is_file():
        return load_split("validation", ROOT / "data" / "raw")
    return pd.DataFrame(demo_rows)

example_direction = (IDIOMS[0], IDIOMS[-1])
examples, cutoffs = add_overlap_buckets(
    make_pairs(validation_data(), *example_direction, max_rows=100, seed=13)
)
display(examples[["src_text", "tgt_text", "overlap", "bucket"]].head(4))
fig, ax = plt.subplots(figsize=(7, 2.7))
ax.hist(examples["overlap"], bins=np.linspace(0, 1, 16), color="#0f766e")
for value, label in zip(cutoffs.values(), ["33rd percentile", "67th percentile"]):
    ax.axvline(value, linestyle="--", label=label)
ax.set(title=f"Lexical overlap · {data_label}", xlabel="Jaccard similarity", ylabel="Pairs")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Play with a live retrieval query
From the repository root, run `python scripts/fetch_mediomatix.py --splits validation` once to switch from the offline demo to real validation data. Choose a direction, a candidate pool size and sampling seed, and a query index within that pool. The table marks the aligned target so you can see when a plausible-looking alternative ranks ahead of it.

For MiniLM or pretrained XLM-R, install `requirements-embeddings.txt`. The first run downloads weights. To try the fine-tuned model live, train and save a checkpoint with [the fine-tuning notebook](finetuning_mediomatix.ipynb), then enter the path to its Hugging Face model directory. The playground applies the same masked mean pooling and 128-token limit as the fine-tuning notebook. Playground ranks use validation candidates. Section 3 shows the separate held-out benchmark.

In [ ]:
@lru_cache(maxsize=2)
def pretrained_model(model_name):
    return load_embedding_model(model_name)

@lru_cache(maxsize=1)
def fine_tuned_model(model_path):
    from sentence_transformers import SentenceTransformer, models
    encoder = models.Transformer(model_path, max_seq_length=128)
    pooling = models.Pooling(encoder.get_word_embedding_dimension(), pooling_mode="mean")
    model = SentenceTransformer(modules=[encoder, pooling])
    model.eval()
    return model

@lru_cache(maxsize=4)
def retrieval_pool(source, target, pool_size, seed, method, checkpoint):
    pairs = make_pairs(validation_data(), source, target, max_rows=pool_size, seed=seed)
    if pairs.empty:
        raise ValueError("No aligned rows are available for this direction.")
    queries, candidates = pairs["src_text"].tolist(), pairs["tgt_text"].tolist()
    if method == "TF-IDF":
        q, c = tfidf_vectors(queries, candidates)
    elif method == "Fine-tuned XLM-R":
        q, c = embedding_vectors(queries, candidates, checkpoint,
                                 model=fine_tuned_model(checkpoint))
    else:
        name = DEFAULT_MODEL if method == "MiniLM" else XLM_R_MODEL
        q, c = embedding_vectors(queries, candidates, name, model=pretrained_model(name))
    return pairs, q, c

def retrieval_view(direction, pool_size, seed, query, method, checkpoint, top_k):
    source, target = direction
    model_path = ""
    if method == "Fine-tuned XLM-R":
        if not checkpoint.strip():
            print("Enter the directory of a saved fine-tuned checkpoint.")
            return
        model_dir = Path(checkpoint.strip()).expanduser()
        if not model_dir.is_absolute():
            model_dir = ROOT / model_dir
        if not (model_dir / "config.json").is_file():
            print("Choose a local checkpoint directory containing config.json.")
            return
        model_path = str(model_dir.resolve())
    try:
        pairs, q, c = retrieval_pool(source, target, pool_size, seed, method, model_path)
    except (FileNotFoundError, RuntimeError, ValueError, ImportError, OSError) as exc:
        print(exc)
        return
    if query >= len(pairs):
        print(f"Choose a query index from 0 to {len(pairs) - 1}.")
        return
    scores = q[query:query + 1] @ c.T
    scores = np.asarray(scores.toarray() if hasattr(scores, "toarray") else scores).ravel()
    gold = int(pairs.iloc[query]["gold"])
    gold_rank = 1 + np.count_nonzero(scores > scores[gold])
    order = np.argsort(-scores, kind="stable")[:top_k]
    print(f"{data_label} · {source} → {target} · {method}")
    print("Query:", pairs.iloc[query]["src_text"])
    print(f"Aligned target (rank {gold_rank}, ties share the best rank):",
          pairs.iloc[gold]["tgt_text"])
    fig, ax = plt.subplots(figsize=(7, 2.5))
    labels = [f"Candidate {j}" + (" (aligned)" if j == gold else "") for j in order]
    ax.barh(labels[::-1], scores[order][::-1], color=["#0f766e" if j == gold else "#93c5fd" for j in order[::-1]])
    ax.set(xlabel="Cosine similarity", title="Top candidate scores")
    plt.tight_layout()
    plt.show()
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame({"cosine": scores[order], "aligned_match": order == gold,
                              "candidate": pairs.iloc[order]["tgt_text"].to_numpy()}))

directions = [(f"{a} → {b}", (a, b)) for a in IDIOMS for b in IDIOMS if a != b]
display(widgets.interactive(
    retrieval_view, {"manual": True, "manual_name": "Rank candidates"},
    direction=widgets.Dropdown(options=directions),
    pool_size=widgets.Dropdown(options=[20, 100, 500], value=100),
    seed=widgets.BoundedIntText(value=13, min=0, max=2**32 - 1),
    query=widgets.BoundedIntText(value=0, min=0, max=499),
    method=widgets.Dropdown(options=["TF-IDF", "MiniLM", "Pretrained XLM-R", "Fine-tuned XLM-R"]),
    checkpoint=widgets.Text(placeholder="Fine-tuned checkpoint directory"),
    top_k=widgets.IntSlider(min=1, max=10, value=5),
))

In [ ]:
retrieval_view(example_direction, 100, 13, 0, "TF-IDF", "", 5)

## 5. Move from one match to a whole candidate pool

Recall@1 counts exact first-place matches. Mean reciprocal rank also gives partial credit when the aligned target ranks lower. This chart recomputes TF-IDF on the selected example direction. The next section uses the team's held-out benchmark across all directions.

In [ ]:
_, q, candidates = retrieval_pool(*example_direction, 100, 13, "TF-IDF", "")
ranks = ranks_from_vectors(q, candidates, examples["gold"].to_numpy())
print(f"Recall@1: {(ranks == 1).mean():.1%}  |  MRR: {(1/ranks).mean():.3f}")
fig, ax = plt.subplots(figsize=(7, 2.6))
examples.assign(hit=ranks == 1).groupby("bucket")["hit"].mean().reindex(
    ["low", "mid", "high"]
).plot.bar(ax=ax, color="#0f766e", rot=0)
ax.set(ylim=(0, 1), ylabel="Recall@1", xlabel="Overlap group", title=f"TF-IDF · {data_label}")
plt.tight_layout()
plt.show()

## 6. Compare the recorded held-out benchmark
Change the lexical overlap group and metric to compare TF-IDF, pretrained XLM-R, and fine-tuned XLM-R on the same held-out test protocol. The XLM-R scores come from `results/test_summary.csv`. TF-IDF scores are rounded to four decimals from the saved output in the [fine-tuning notebook](finetuning_mediomatix.ipynb). This section runs without downloading models or retraining.

In [ ]:
test_summary = pd.read_csv(ROOT / "results" / "test_summary.csv")

tfidf = pd.DataFrame([
    ("all", 89684, 0.7020, 0.7571),
    ("low", 29654, 0.3507, 0.4336),
    ("mid", 30366, 0.8175, 0.8767),
    ("high", 29664, 0.9348, 0.9580),
], columns=["bucket", "queries", "recall_at_1", "mrr"]).assign(model="TF-IDF", directions=20)
test_summary = pd.concat([test_summary, tfidf], ignore_index=True)

def benchmark_view(bucket, metric):
    rows = test_summary[test_summary["bucket"].eq(bucket)].sort_values(metric, ascending=False)
    values = rows.set_index("model")[[metric, "directions", "queries"]]
    display(values.style.format({metric: "{:.2%}"}).bar(subset=[metric], vmin=0, vmax=1,
                                                        color="#90c4ed"))

display(widgets.interactive(
    benchmark_view,
    bucket=widgets.Dropdown(options=[("All", "all"), ("Low", "low"),
                                     ("Medium", "mid"), ("High", "high")]),
    metric=widgets.Dropdown(options=[("Recall@1", "recall_at_1"), ("MRR", "mrr")]),
))

In [ ]:
order = ["low", "mid", "high"]
chart = test_summary.pivot(index="bucket", columns="model", values="recall_at_1").loc[order]
ax = chart.plot.bar(figsize=(9, 3.4), rot=0, color=["#0f766e", "#93c5fd", "#5046a0"])
ax.set(ylim=(0, 1), ylabel="Recall@1", xlabel="Lexical overlap", title="Recorded held-out retrieval results")
ax.legend(title=None, loc="upper left")
plt.tight_layout()
plt.show()
display(chart.style.format("{:.2%}"))

## 7. Try these comparisons

1. Change only the direction. Look at whether the aligned target moves up or down.
2. Keep the direction and seed fixed. Increase the candidate pool and see whether similar-looking distractors appear.
3. In the held-out results, compare TF-IDF and fine-tuned XLM-R in the low- and high-overlap groups. How does the better model change with overlap?
4. For a live dense comparison on real data, keep the pool and seed fixed and select another encoder. The fine-tuned option needs a local checkpoint.